# 01 — NSL-KDD dataset exploration

This notebook inspects the **real NSL-KDD files** used by the pipeline. It reads the splits and
metadata produced by `python run_pipeline.py --mode prepare`; it does not fabricate or resample data.

Scope of the executed study: **quick profile**, 12,000-row stratified pool from `KDDTrain+`,
70/15/15 split, seed 42. The official `KDDTest+` file is external and is never trained on.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
DATA = ROOT / "data"
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def show_image(path):
    """Render a saved PNG inside the notebook without extra display dependencies."""
    img = PILImage.open(path)
    plt.figure(figsize=(min(12, img.width / 100), min(8, img.height / 100)))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    plt.close()


print("Project root:", ROOT)

## 1. Dataset validation summary

In [ ]:
validation = json.loads((DATA / "metadata" / "dataset_validation.json").read_text(encoding="utf-8"))
summary = {
    "train_rows": validation["train_rows"],
    "test_rows": validation["test_rows"],
    "n_columns": validation["n_columns"],
    "n_features": validation["n_features"],
    "missing_values_train": validation["missing_values_train"],
    "missing_values_test": validation["missing_values_test"],
    "duplicate_rows_train": validation["duplicate_rows_train"],
    "duplicate_rows_test": validation["duplicate_rows_test"],
    "train_attack_types": validation["train_attack_types"],
    "test_attack_types": validation["test_attack_types"],
    "test_only_attack_types": validation["test_only_attack_types"],
    "valid": validation["valid"],
}
summary

**Reading:** the raw files contain 125,973 training records and 22,544 external test records.
There are no missing values and no exact duplicate rows. The external test file contains 17 attack
types that never appear in `KDDTrain+`; these support the external novel-type experiment.

## 2. Split sizes and class balance

In [ ]:
splits = {}
for name in ("train", "validation", "test"):
    df = pd.read_parquet(DATA / "splits" / f"{name}.parquet")
    splits[name] = df
    print(f"{name:10s} rows={len(df):5d}  attack_rate={df['is_attack'].mean():.4f}")

pd.DataFrame({
    name: df["attack_category"].value_counts()
    for name, df in splits.items()
}).fillna(0).astype(int)

The three splits are stratified on the binary label, so attack rates agree to within 0.03
percentage points. R2L and U2R are rare in the pool; this is why the unseen-category experiment
uses a balanced holdout rather than raw prevalence.

## 3. Label distribution

In [ ]:
label_counts = pd.concat(
    [df["attack_type"].value_counts().rename(name) for name, df in splits.items()],
    axis=1,
).fillna(0).astype(int).sort_values("train", ascending=False)
label_counts.head(20)

In [ ]:
top = label_counts.head(15)
ax = top.plot(kind="bar", figsize=(11, 5))
ax.set_yscale("log")
ax.set_ylabel("records (log scale)")
ax.set_title("NSL-KDD attack types by split (top 15, quick pool)")
plt.tight_layout()
plt.show()

`normal`, `neptune` and `satan` dominate. Rare categories such as U2R contain only a handful of
training rows, which limits what any classifier can learn about them. Metrics are therefore always
reported with per-class and per-category breakdowns, not only aggregate accuracy.

## 4. External test-only attack types

In [ ]:
external = pd.read_parquet(DATA / "processed" / "external.parquet")
final = json.loads((RESULTS / "final_summary.json").read_text(encoding="utf-8"))
novel_types = final["external_novel_details"]["novel_types"]
external["is_novel_type"] = external["attack_type"].isin(novel_types)
print("External rows:", len(external))
print("Canonical novel types:", len(novel_types))
print("Novel-type rows:", int(external["is_novel_type"].sum()))
print("Normal rows in external set:", int((external["is_attack"] == 0).sum()))
external.loc[external["is_novel_type"], "attack_type"].value_counts()

The 17 novel types define the external novel-type test. Novelty is judged against the **full
KDDTrain+ label set**, not the 12,000-row quick pool. That distinction matters: judging against the
subsample would wrongly label some common attack types as novel.

## 5. Reproducibility check

In [ ]:
meta = json.loads((DATA / "splits" / "split_metadata.json").read_text(encoding="utf-8"))
meta

**Takeaways**

1. The data are real, complete and split without row overlap.
2. The quick pool preserves the binary attack rate of the training file.
3. Severe label imbalance and 17 external novel types make per-category reporting mandatory.
4. All later notebooks reuse these exact splits and the saved reduced matrices.